In [ ]:
import sys
sys.path.append('..')
import MeshFEM
import mesh, inflation, numpy as np, importlib, fd_validation, visualization, parametric_pillows, wall_generation, py_newton_optimizer
import utils, sheet_optimizer
from numpy.linalg import norm

In [ ]:
m, fuseMarkers, edgeSegments = wall_generation.triangulate_channel_walls(*parametric_pillows.concentricCircles(8, 50), 0.001)
isheet = inflation.InflatableSheet(m, np.array(fuseMarkers) != 0)

In [ ]:
visualization.plot_2d_mesh(m, width=12, height=12, pointList=isheet.wallVertices())

In [ ]:
targetSurf = mesh.Mesh('data/pringle.obj')
targetAttractedSheet = inflation.TargetAttractedInflation(isheet, targetSurf)
targetAttractedSheet.fittingWeight = 0.1

In [ ]:
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-7
opts.niter = 50

In [ ]:
isheet.pressure = 30
inflation.inflation_newton(targetAttractedSheet, [], opts)

In [ ]:
rso = inflation.ReducedSheetOptimizer(targetAttractedSheet, [], opts)

In [ ]:
from tri_mesh_viewer import TriMeshViewer
view = TriMeshViewer(targetAttractedSheet.sheet().visualizationMesh())
view.show()

In [ ]:
view.showWireframe()

In [ ]:
rso.useFirstOrderPrediction = True

In [ ]:
xorig = rso.getVars()

In [ ]:
rso.setVars(xorig + 3.5e-3 * np.random.uniform(low=-1, high=1, size=xorig.shape))

In [ ]:
rso.energy(etype=rso.EnergyType.CollapseBarrier)

In [ ]:
rso.wallSmoothingWeight = 1
rso.boundarySmoothingWeight = 1.0

In [ ]:
rso.energy(etype=rso.EnergyType.Smoothing)

In [ ]:
rso.energy(etype=rso.EnergyType.Full)

In [ ]:
rso.commitDesign()
fd_validation.validateGrad(rso, fd_eps=1e-6, etype=rso.EnergyType.Full)

In [ ]:
rso.commitDesign()
fd_validation.validateGrad(rso, fd_eps=1e-6, etype=rso.EnergyType.Smoothing)

In [ ]:
fd_validation.gradConvergencePlot(rso, energyType=rso.EnergyType.CollapseBarrier)

In [ ]:
# Validate the accelerated formula matches to machine precision
testVec = np.random.normal(size=2 * isheet.mesh().numVertices()).reshape(-1, 2)
accel = rso.apply_d2E_dxdX(testVec)
unaccel = rso.apply_d2E_dxdX_unaccelerated(testVec)
np.linalg.norm(accel - unaccel) / np.linalg.norm(unaccel)

In [ ]:
inflation.benchmark_reset()
fd_validation.gradConvergencePlot(rso, energyType=rso.EnergyType.Fitting)
inflation.benchmark_report()

In [ ]:
inflation.benchmark_reset()
fd_validation.gradConvergencePlot(rso, energyType=rso.EnergyType.Full)
inflation.benchmark_report()

### Finite difference test of intermediate configuration

In [ ]:
import sys
sys.path.append('..')
import MeshFEM
import mesh, inflation, numpy as np, importlib, fd_validation, visualization, parametric_pillows, wall_generation, py_newton_optimizer
import utils, sheet_optimizer
from numpy.linalg import norm

In [ ]:
sheet_opt = sheet_optimizer.load('data/sheet_opt.pkl.gz')
rso = sheet_opt.rso

In [ ]:
utils.allEnergies(rso)

In [ ]:
fd_validation.gradConvergencePlot(rso, epsilons=np.logspace(-9, -4, 30), customArgs={'etype': rso.EnergyType.CollapseBarrier, 'bailEnergyThreshold': 0.0})

In [ ]:
for t in range(rso.mesh().numTris()):
    cpe = rso.collapseBarrier().collapsePreventionEnergy(t)
    cpe.applyStretchBarrier = True
    cpe.stretchBarrierActivation = 0.9

In [ ]:
utils.allEnergies(rso)

In [ ]:
fd_validation.gradConvergencePlot(rso, epsilons=np.logspace(-9, -4, 30), customArgs={'etype': rso.EnergyType.CollapseBarrier, 'bailEnergyThreshold': 0.0})

In [ ]:
visualization.designAlterationStretchHistograms(rso)